# BaseLine `Zero-shot image classification`

## Description

__Цель__  - изучение особенностей использования `Zero-shot learning` и помпт-инженеринга в задачах классификация изображений.

__Задачи__ - 
* собрать pipeline для максимально точной классификации изображений в заданном наборе данных.
* достигнуть максимальной точности классификации



__Задача__ [`Zero-shot` Классификация изображений](https://huggingface.co/docs/transformers/en/tasks/zero_shot_image_classification) - это задача, которая включает в себя классификацию изображений по различным категориям с использованием модели, которая не была явно обучена непосредственно на релевантных  данных, содержащих помеченные примеры из целевых категорий.

Традиционно классификация изображений требует обучения модели на определенном наборе размечченных изображений, и эта модель учится “сопоставлять” определенные особенности изображения с надписями. Когда возникает необходимость использовать такую модель для задачи классификации, которая вводит новый набор меток, требуется тонкая настройка, чтобы “перекалибровать” модель.

В отличие от этого, модели `Zero-shot image classification` как правило, являются мультимодальными моделями, которые были обучены на избыточно-большом наборе изображений и связанных с ними описаний. Эти модели выучивают согласованные представления текстов и элементов изображений на визуальном языке, которые могут использоваться для широкого круга задач компьютерного зрения, включая классификацию изображений без дообучения.

Таким образом, `Zero-shot image classification` - это более гибкий подход к классификации изображений, который позволяет моделям обобщаться на новые категории без необходимости в дополнительных обучающих данных и позволяет пользователям решать `image-to-text` (изображения с текстовыми описаниями) для целевых объектов в достаточно произвольной форме.

Набор данных содержит 2 директории
* `train`
* `test`

В каждой  директории `1413` изображений, разделенных на `353` класса.

также в каждой директории можно найти файл разметки `label.csv`. В директории `train` файл содержит:\
* `ID` - название файла в `train`, 
* `class` - название класса, 
* `group`- группа класса, 
* `language` - язык, на котором класс задан. 

В директории `test` столбец `class` не содержится.

Общий список классов:

## Import



In [41]:
from pathlib import Path
from transformers import pipeline
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
import os


## Загрузка данных

Директории с данными и название файла для подачи результатов

In [42]:
IMAGE_FOLDER_TRAIN = "train"
IMAGE_FOLDER_TEST = "test"
OUTPUT_CSV = "submit.csv"

Файл разметки.

In [43]:
df_train = pd.read_csv(os.path.join(IMAGE_FOLDER_TRAIN,'labels.csv'))
df_train.head(3)

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,class,group,language,ID
0,0,0,0,Elefante,Mamífero,Español,1.jpg
1,1,1,1,bos-gaurus,Animalia,Latina,2.jpg
2,2,2,2,martes-americana,Animalia,Latina,3.jpg


In [44]:
df_test = pd.read_csv(os.path.join(IMAGE_FOLDER_TEST,'labels.csv'))
df_test.head(3)

,group,language,ID
0,Dishes,English,1.jpg
1,Animalia,Latina,2.jpg
2,Dishes,English,3.jpg


## Классификация 

зададим промпты для классификации как названия классов

In [58]:
def make_zs_prompts(df):
    return df['class'].unique()

prompts4clf =   make_zs_prompts(df_train)  
print(len(np.unique(prompts4clf)))

352


и так для каждого из 352 классов промпт готов.

Выберем одну из доступных моделей семейства `CLIP`

In [46]:
classifier = pipeline(
    "zero-shot-image-classification",
    model="omarques/clip-vit-base-patch32-demo",
    use_fast=True,  # Explicitly use fast tokenizer/processor
    device=0  # Uses cuda:0 if available
)

Device set to use cuda:0


Проведем классификацию  изображений

In [47]:
def classify_images(image_folder, classifier, candidate_labels):
    """
    Классифицирует изображения в указанной папке с помощью заданного классификатора.

    Параметры:
        image_folder (str или Path): Путь к папке с изображениями.
        classifier: Функция или модель, принимающая изображение и список меток.
                    Должна возвращать список словарей с ключами "label" и "score".
        candidate_labels (list): Список возможных меток для классификации.

    Возвращает:
        pd.DataFrame: Таблица с колонками "ID" (имя файла) и "predicted_class".
    """
    results = []
    image_paths = [
        p for p in Path(image_folder).iterdir()
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    ]

    for img_path in tqdm(image_paths, desc="Classifying"):
        image = Image.open(img_path).convert("RGB")
        preds = classifier(image, candidate_labels=candidate_labels)
        top_pred = preds[0]["label"]
        results.append({"ID": img_path.name, "predicted_class": top_pred})

    return pd.DataFrame(results)

In [48]:
results_df = classify_images(
    image_folder=IMAGE_FOLDER_TRAIN,
    classifier=classifier,
    candidate_labels=prompts4clf
)

Classifying: 100%|█████████████████████████████████████████████████████████████████| 1412/1412 [01:57<00:00, 11.99it/s]


Проверим результат

In [49]:
merged_df = pd.merge(df_train, results_df, on='ID', how='outer')
merged_df['is_correct'] = merged_df.apply(lambda row: row['class'] in row['predicted_class'], axis=1)
accuracy = merged_df['is_correct'].mean()
accuracy

np.float64(0.6055240793201133)

Проведем анализ ошибок классификации

In [50]:
merged_df[merged_df['is_correct']==False]

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,class,group,language,ID,predicted_class,is_correct
0,0,0,0,Elefante,Mamífero,Español,1.jpg,mammuthus-primigeniu,False
1,9,9,9,desmodus-rotundus,Animalia,Latina,10.jpg,rattus-rattus,False
2,99,99,99,ibis,Other,English,100.jpg,ornithorhynchus-anatinus,False
5,1001,1001,1001,cryptoprocta-ferox,Animalia,Latina,1002.jpg,sciurus-carolinensis,False
7,1003,1003,1003,phoebetria-fusca,Animalia,Latina,1004.jpg,colaptes-auratus,False
...,...,...,...,...,...,...,...,...,...
1393,981,981,981,phascolarctos-cinereus,Animalia,Latina,982.jpg,macropus-giganteus,False
1394,982,982,982,ovis-aries,Animalia,Latina,983.jpg,struthio-camelus,False
1397,985,985,985,crayfish,Other,English,986.jpg,lobster,False
1399,987,987,987,dermochelys-coriacea,Animalia,Latina,988.jpg,hawksbill,False


# Оформление `submit.csv`

Классификация тестовых данных

In [51]:
results_df = classify_images(
    image_folder=IMAGE_FOLDER_TEST,
    classifier=classifier,
    candidate_labels=prompts4clf
)

Classifying: 100%|█████████████████████████████████████████████████████████████████| 1412/1412 [01:57<00:00, 12.05it/s]


Проверим результаты

In [52]:
results_df.head(3)

,ID,predicted_class
0,1.jpg,ice_cream
1,10.jpg,airplanes
2,100.jpg,ice_cream


Сохронение 

In [53]:
results.to_csv(OUTPUT_CSV, index = False)
OUTPUT_CSV

'submit.csv'

# Пример того, как будет проводится тест

In [54]:
submit = 'submit.csv'
keys = 'test_with_answers.csv'

In [55]:
results = pd.read_csv(submit, index_col = False)
df = pd.read_csv(keys, index_col = False)

In [56]:
merged_df = pd.merge(df, pd.DataFrame(results), on='ID', how='outer')

In [57]:
merged_df['is_correct'] = merged_df.apply(lambda row: row['class'] in row['predicted_class'], axis=1)
accuracy = merged_df['is_correct'].mean()
accuracy

np.float64(0.6019830028328612)